In [1]:
from ngsolve import *
import netgen.gui
import time

optfile ./ng.opt does not exist - using default values
togl-version : 2
OCC module loaded
loading ngsolve library
NGSolve-6.2.2105
Using Lapack
Including sparse direct solver Pardiso
Running parallel using 8 thread(s)


## 2维曲面生成

In [48]:
from scipy.sparse import *
import numpy as np

def MyInv(Amat, Vec, FreeDofs:np.ndarray=None):
    '''
        # A is a ngsolve matrix，例如可以如下生成
        A = la.SparseMatrixd.CreateFromCOO([0,1,2], [0,1,2], [1,2,3], 3, 3)
        MyInv(A,BaseVector(np.array([1,2,3])),np.array([1,1,0],dtype=bool))
        gfu.vec.data += BaseVector(MyInv(a.mat,res:BaseVector,np.array(X.FreeDofs())))
        其中FreeDofs的dtype需要时bool才可以
    '''
    if FreeDofs is None:
        FreeDofs = np.array(np.ones(Vec.FV().NumPy().shape),dtype=bool)
    numFree = np.sum(FreeDofs)
    A_data = list(Amat.COO())
    
    A_coo = coo_matrix((A_data[2].NumPy(),(np.array(A_data[0]),np.array(A_data[1]))),Amat.shape)
    A_csr = A_coo.tocsr()
    A_new_csr = A_csr[FreeDofs][:, FreeDofs]
    b = Vec.FV().NumPy()[FreeDofs]
    
    # 使用spsolve求解Ax = b
    x = linalg.spsolve(A_new_csr, b)
    res = np.zeros(FreeDofs.shape)
    res[FreeDofs] = x
    return res

In [49]:
from netgen.csg import Pnt,SplineCurve2d,CSGeometry,Revolution,Sphere
import numpy as np
import netgen.meshing as ngm
from netgen.csg import *
from netgen.meshing import MeshingStep

In [ ]:
from netgen.occ import SplineApproximation, Pnt, Axis, Face, Wire, Segment, Revolve, OCCGeometry, Z
#%% Construction of Initial Curved Mesh
dim = 3
tau0 = 0.005
vtknum = 30
ParamG_val = 10
order = 1
print('spatial order is {}'.format(order))
tauval = tau0
msize = 0.04
T0 = 0  # Start from a critical time point
T = 0.6 # End at T

num_a = 0.1 + 0.05*np.sin(2*np.pi*T0)
num_L = 1 + 0.2*np.sin(4*np.pi*T0)
def num_G(xx): return ParamG_val*xx*(xx-(ParamG_val-1)/ParamG_val)

def Curve(z): 
    if z == -num_L or z == num_L:
        res = Pnt(0, 0, z)
    else:
        res = Pnt(num_a*sqrt(1-num_G(z**2/num_L**2)), 0, z)
    return res

n = 100
pnts = [Curve(z) for z in num_L*np.cos(np.linspace(0,np.pi,100))]
spline = SplineApproximation(pnts, tol=1e-4)
seg3 = Segment(Pnt(0, 0, num_L), Pnt(0, 0, -num_L))
f = Face(Wire([spline,seg3]))
torus = f.Revolve(Axis((0,0,0), Z), 360)
geo = OCCGeometry(torus)
mesh = Mesh(OCCGeometry(torus).GenerateMesh(maxh=msize,
            perfstepsend=ngm.MeshingStep.MESHSURFACE))

## 有限元空间
* 标量空间 for multiplier
* 向量空间 for position and velocity

In [ ]:
fes = H1(mesh,order=1)
fesV = VectorH1(mesh,order=1)
fesMix = fes*fesV

## 有限元函数
* 位置 X
* 位移 Disp

In [72]:
Disp = GridFunction(fesV)
X_old = GridFunction(fesV)
Solution = GridFunction(fesMix)

## 设定位置X的初值

In [73]:
Vertices_Coords = np.array([v.point for v in mesh.vertices])
X_old.vec.data = BaseVector(Vertices_Coords.flatten('F'))

## Mass lumping on surface

In [74]:
ir = IntegrationRule(points = [(0,0), (1,0), (0,1)], weights = [1/6, 1/6, 1/6])
ds_lumping = ds(intrules = { TRIG : ir })

## Time stepping parameter

In [75]:
tauval = 0.001
t_old = 0

## 显示网格移动的设定

In [76]:
SetVisualization(deformation=True)

In [77]:
Draw(Disp,mesh,'disp',deformation=True)

## BGN弱形式
* BGN本身求解的是新时刻的X；这里求解的是D = X^n+1 - X^n

In [68]:
kappa, D = fesMix.TrialFunction()
chi, eta = fesMix.TestFunction()

lhs = BilinearForm(fesMix)
lhs += InnerProduct(D,specialcf.normal(3))*chi*ds_lumping + tauval*InnerProduct(grad(kappa).Trace(),grad(chi).Trace())*ds
lhs += -InnerProduct(eta,specialcf.normal(3))*kappa*ds_lumping + InnerProduct(grad(D).Trace(),grad(eta).Trace())*ds

rhs = LinearForm(fesMix)
rhs += -InnerProduct(grad(X_old).Trace(),grad(eta).Trace())*ds

## 时间演化

In [69]:
t_old = 0
while t_old<0.006:
    mesh.SetDeformation(Disp)
    lhs.Assemble()
    rhs.Assemble()
    # Solution.vec.data = lhs.mat.Inverse(inverse="pardiso")*rhs.vec
    Solution.vec.data = BaseVector(MyInv(lhs.mat, rhs.vec))
    
    Disp.vec.data = BaseVector(Disp.vec.FV().NumPy() + Solution.components[1].vec.FV().NumPy())
    X_old.vec.data = BaseVector(X_old.vec.FV().NumPy() + Solution.components[1].vec.FV().NumPy())
    Redraw()
    # time.sleep(0.001)
    t_old += tauval